# ST-OMR Meter V5-1 — 30 TRAIN BBox Pilot

**Güvenlik sınırı:** Bu notebook yalnız `METER_V2_1500_PACKAGE_AB_CLEAN` veri setini kabul eder. Önce 1500 yapısal/manifest gate'i çalışır, ardından `final_holdout` kilitlenir. Annotation UI yalnız TRAIN içinden deterministik 10×2/4 + 10×3/4 + 10×4/4 örneği açar. Original `image.png` dosyaları değiştirilmez; annotation ayrı CSV'ye checkpoint edilir. Training, tuning, checkpoint/model loading ve inference bu notebookta yoktur.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, pathlib, subprocess, sys, shutil
EXPECTED_CODE_SHA = '0146ecae4f3abb06175872bebc6b7f15644b4773'
BRANCH = 'fix/meter-v5-1-clean-bbox-pilot'
REPO_URL = 'https://github.com/khfy7wpr5p-maker/st-omr-training.git'
REPO_DIR = pathlib.Path('/content/st-omr-training-v5-1')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(['git','clone','--depth','20','--branch',BRANCH,REPO_URL,str(REPO_DIR)], check=True)
subprocess.run(['git','-C',str(REPO_DIR),'checkout','--detach',EXPECTED_CODE_SHA], check=True)
actual = subprocess.check_output(['git','-C',str(REPO_DIR),'rev-parse','HEAD'], text=True).strip()
assert actual == EXPECTED_CODE_SHA, (actual, EXPECTED_CODE_SHA)
subprocess.run([sys.executable,'-m','pip','install','-q','Pillow==12.3.0'], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print('CODE_PIN=PASS', actual)


## Precheck — dataset discovery + 1500 gate + final_holdout lock

Bu hücre `/content/drive/MyDrive` altında exact `METER_V2_1500_PACKAGE_AB_CLEAN` adını otomatik arar. 0 veya birden fazla eşleşmede fail-closed durur. Final holdout görüntülerini display/decode etmez; yalnız klasör/sayı/manifest ve `image.png` varlık kapısını doğrular.


In [ ]:
from st_omr_training.meter_v5_1_bbox_pilot import (
    discover_data_root, verify_dataset_structure, ensure_final_holdout_lock,
)
DATA_ROOT = discover_data_root('/content/drive/MyDrive')
GATE = verify_dataset_structure(DATA_ROOT)
LOCK_PATH = ensure_final_holdout_lock(DATA_ROOT, GATE)
print('DATASET_GATE=PASS')
print('data_root=', DATA_ROOT)
print('total=', GATE['total'])
print('unique_family_id=', GATE['unique_family_id'])
print('unique_sample_id=', GATE['unique_sample_id'])
print('unique_source_image=', GATE['unique_source_image'])
print('package_ab_only=', GATE['package_ab_only'])
print('cross_split_family_leakage=', GATE['cross_split_family_leakage'])
print('cross_meter_family_overlap=', GATE['cross_meter_family_overlap'])
for split in ('train','val','final_holdout'):
    print(split.upper(), '2/4=', GATE['directory_counts'][f'{split}/2_4'],
          '3/4=', GATE['directory_counts'][f'{split}/3_4'],
          '4/4=', GATE['directory_counts'][f'{split}/4_4'])
print('final_holdout_locked=', GATE['final_holdout_locked'])
print('final_holdout_lock=', LOCK_PATH)
print('ANNOTATION_SCOPE=train_pilot_30_only')
print('MODEL_OPENED=False; TRAINING=False; TUNING=False; INFERENCE_COUNT=0')


## 30 TRAIN interaktif BBox pilotu

Precheck `DATASET_GATE=PASS` verdiyse bu hücreyi çalıştır. Mouse ile **tek kutu** çiz: üst+alt meter rakamlarının tamamını kapsa; clef/key signature/ilk nota mümkün olduğunca dışarıda kalsın. `KAYDET VE SONRAKİ` PASS olarak checkpoint eder. Emin olmadığın örnekte `REVIEW / ATLA` kullan. Her kayıt anında Drive'daki `annotations/bbox_pilot_30.csv` dosyasına atomik checkpoint edilir ve runtime kapanırsa sample_id üzerinden devam eder.


In [ ]:
from st_omr_training.meter_v5_1_bbox_pilot_colab import launch_colab_pilot
SESSION = launch_colab_pilot(data_root=str(DATA_ROOT))
print('PILOT_UI=READY')
print('resume_index=', SESSION.resume_index())
print('handled=', SESSION.handled_count, 'pass=', SESSION.pass_count, 'review=', SESSION.review_count)
print('FINAL_HOLDOUT=LOCKED; TRAINING=False; MODEL_OPENED=False; INFERENCE_COUNT=0')


## Pilot audit — yalnız 30 örnek tamamlandıktan sonra

UI'da 30/30 işlendikten sonra bu hücreyi çalıştır. Audit insan bbox'larını değiştirmez; yalnız mekanik kontrolleri ve boyut istatistiklerini raporlar.


In [ ]:
import json
from st_omr_training.meter_v5_1_bbox_pilot import write_pilot_audit
AUDIT_PATH = write_pilot_audit(DATA_ROOT)
AUDIT = json.loads(AUDIT_PATH.read_text(encoding='utf-8'))
print(json.dumps(AUDIT, indent=2, sort_keys=True))
print('AUDIT_PATH=', AUDIT_PATH)
print('FINAL_HOLDOUT=LOCKED; TRAINING=False; MODEL_OPENED=False; INFERENCE_COUNT=0')
